<a href="https://colab.research.google.com/github/ofir2207/Cloud-project/blob/main/ex11_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai anthropic
!pip install -q langchain langchain-community langchain-openai
!pip install -q chromadb sentence-transformers
!pip install -q faiss-cpu
!pip install -q numpy pandas matplotlib
print("\n✓ All libraries installed successfully!\n")

import random
import time
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum
import numpy as np

## Simple Reflex Agent
- React to current percepts only
- Follow condition-action rules
- No memory of past events

Example: Thermostat, Traffic Light

In [ ]:
class ThermostatAgent:
    """
    A simple reflex agent that controls heating based on temperature.
    """
    def __init__(self, target_temp: float = 22.0):
        self.target_temp = target_temp
        self.heater_on = False

    def perceive_and_act(self, current_temp: float) -> str:
        """React to current temperature reading."""
        if current_temp < self.target_temp and not self.heater_on:
            self.heater_on = True
            return f"🔥 Temperature {current_temp}°C is below target {self.target_temp}°C. Turning heater ON."
        elif current_temp >= self.target_temp and self.heater_on:
            self.heater_on = False
            return f"✓ Temperature {current_temp}°C reached target {self.target_temp}°C. Turning heater OFF."
        else:
            status = "ON" if self.heater_on else "OFF"
            return f"→ Temperature {current_temp}°C. Heater status: {status}. No action needed."

In [ ]:
# Demo: Simple Reflex Agent
print("\n--- Thermostat Agent Demo ---")
thermostat = ThermostatAgent(target_temp=22.0)
temperatures = [18.0, 19.5, 21.0, 22.0, 22.5, 21.5, 20.0]
for temp in temperatures:
    action = thermostat.perceive_and_act(temp)
    print(action)
    time.sleep(0.5)

## Model-Based Reflex Agent
- Maintain internal state/model
- Track world changes over time
- Use history to make better decisions

In [ ]:
class RobotNavigator:
    """
    A model-based reflex agent that navigates a grid while remembering visited locations.
    """
    def __init__(self, grid_size: int = 5):
        self.grid_size = grid_size
        self.position = [0, 0]
        self.visited = set()
        self.visited.add(tuple(self.position))
        self.obstacles = set()

    def perceive_obstacle(self, obstacle_pos: tuple):
        """Update internal model with obstacle information."""
        self.obstacles.add(obstacle_pos)
        print(f"🚧 Obstacle detected and remembered at {obstacle_pos}")

    def move(self, direction: str) -> str:
        """Move in a direction, considering internal model."""
        moves = {
            'up': [0, 1],
            'down': [0, -1],
            'left': [-1, 0],
            'right': [1, 0]
        }

        if direction not in moves:
            return "Invalid direction"

        new_pos = [
            self.position[0] + moves[direction][0],
            self.position[1] + moves[direction][1]
        ]

        if not (0 <= new_pos[0] < self.grid_size and 0 <= new_pos[1] < self.grid_size):
            return f"❌ Cannot move {direction}: out of bounds"

        if tuple(new_pos) in self.obstacles:
            return f"❌ Cannot move {direction}: obstacle remembered at {tuple(new_pos)}"

        self.position = new_pos
        is_new = tuple(self.position) not in self.visited
        self.visited.add(tuple(self.position))

        status = "new location ✨" if is_new else "previously visited"
        return f"→ Moved {direction} to {self.position} ({status})"

In [ ]:
# Demo: Model-Based Reflex Agent
print("\n--- Robot Navigator Demo ---")
robot = RobotNavigator(grid_size=5)
robot.perceive_obstacle((1, 1))
robot.perceive_obstacle((2, 0))
commands = ['right', 'right', 'up', 'left', 'left', 'up', 'right']
for cmd in commands:
    result = robot.move(cmd)
    print(result)
    time.sleep(0.5)
print(f"\n📊 Total locations visited: {len(robot.visited)}")
print(f"📍 Final position: {robot.position}")

## Goal-Based Agent
- Choose actions to achieve specific goals
- Plan ahead and consider future consequences
- Use reasoning to select optimal path

In [ ]:
class PathPlannerAgent:
    """
    A goal-based agent that plans a path to reach a destination.
    """
    def __init__(self, grid_size: int = 5):
        self.grid_size = grid_size
        self.position = [0, 0]
        self.goal = None
        self.obstacles = set()

    def set_goal(self, goal_position: List[int]):
        """Set the goal position."""
        self.goal = goal_position
        print(f"🎯 Goal set to: {self.goal}")

    def add_obstacle(self, obstacle_pos: tuple):
        """Add an obstacle to avoid."""
        self.obstacles.add(obstacle_pos)

    def plan_path(self) -> List[str]:
        """Plan a path to reach the goal using simple A* approach."""
        if not self.goal:
            return []

        path = []
        current = self.position.copy()

        while current != self.goal:
            if current[0] < self.goal[0]:
                next_pos = [current[0] + 1, current[1]]
                if tuple(next_pos) not in self.obstacles:
                    path.append('right')
                    current = next_pos
                    continue
            elif current[0] > self.goal[0]:
                next_pos = [current[0] - 1, current[1]]
                if tuple(next_pos) not in self.obstacles:
                    path.append('left')
                    current = next_pos
                    continue
            if current[1] < self.goal[1]:
                next_pos = [current[0], current[1] + 1]
                if tuple(next_pos) not in self.obstacles:
                    path.append('up')
                    current = next_pos
                    continue
            elif current[1] > self.goal[1]:
                next_pos = [current[0], current[1] - 1]
                if tuple(next_pos) not in self.obstacles:
                    path.append('down')
                    current = next_pos
                    continue
            break
        return path

    def execute_plan(self, plan: List[str]) -> bool:
        """Execute the planned path."""
        print(f"\n📋 Executing plan: {' → '.join(plan)}")

        for i, move in enumerate(plan):
            moves = {'up': [0, 1], 'down': [0, -1], 'left': [-1, 0], 'right': [1, 0]}
            self.position[0] += moves[move][0]
            self.position[1] += moves[move][1]
            print(f"  Step {i+1}: Moved {move} to {self.position}")
            time.sleep(0.3)

        reached_goal = self.position == self.goal
        if reached_goal:
            print(f"✓ Goal reached at {self.position}!")
        else:
            print(f"❌ Failed to reach goal. Current position: {self.position}")

        return reached_goal

In [ ]:
# Demo: Goal-Based Agent
print("\n--- Path Planner Agent Demo ---")
planner = PathPlannerAgent(grid_size=5)
planner.add_obstacle((1, 1))
planner.add_obstacle((2, 1))
print("🚧 Obstacles added at: (1,1) and (2,1)")
planner.set_goal([3, 2])
plan = planner.plan_path()
print(f"\n🗺️  Planned path has {len(plan)} steps")
planner.execute_plan(plan)

## Utility-Based Agent
- Maximize utility/performance measure
- Handle conflicting goals
- Consider tradeoffs between multiple objectives

In [ ]:
class RouteOptimizerAgent:
    """
    A utility-based agent that optimizes route selection based on multiple factors.
    """
    def __init__(self):
        self.preferences = {
            'time': 0.4,
            'cost': 0.3,
            'comfort': 0.3
        }

    def calculate_utility(self, route: Dict[str, float]) -> float:
        """
        Calculate utility score for a route.
        Higher score = better route
        """
        time_score = 1 - (route['time'] / 120)
        cost_score = 1 - (route['cost'] / 50)
        comfort_score = route['comfort'] / 10

        utility = (
            self.preferences['time'] * time_score +
            self.preferences['cost'] * cost_score +
            self.preferences['comfort'] * comfort_score
        )

        return utility

    def select_best_route(self, routes: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Select the route with highest utility."""
        best_route = None
        best_utility = -1

        print("\n🔍 Evaluating routes:\n")
        for route in routes:
            utility = self.calculate_utility(route)
            route['utility'] = utility
            print(f"  {route['name']:15} | Time: {route['time']:3.0f}min | "
                  f"Cost: ${route['cost']:4.1f} | Comfort: {route['comfort']}/10 | "
                  f"Utility: {utility:.3f}")

            if utility > best_utility:
                best_utility = utility
                best_route = route

        return best_route

In [ ]:
# Demo: Utility-Based Agent
print("\n--- Route Optimizer Agent Demo ---")
optimizer = RouteOptimizerAgent()
routes = [
    {'name': 'Highway',      'time': 30,  'cost': 15.0, 'comfort': 7},
    {'name': 'Scenic Route', 'time': 60,  'cost': 10.0, 'comfort': 9},
    {'name': 'City Streets', 'time': 45,  'cost': 5.0,  'comfort': 4},
    {'name': 'Express',      'time': 25,  'cost': 25.0, 'comfort': 8},
]
best = optimizer.select_best_route(routes)
print(f"\n✓ Best route selected: {best['name']} (Utility: {best['utility']:.3f})")

## Learning Agent
- Improve performance through experience
- Adapt to new environments
- Use reinforcement learning principles

In [ ]:
class QLearningAgent:
    """
    A learning agent that uses Q-Learning to learn optimal actions.
    """
    def __init__(self, states: int, actions: int, learning_rate: float = 0.1,
                 discount_factor: float = 0.9, exploration_rate: float = 0.3):
        self.q_table = np.zeros((states, actions))
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = exploration_rate
        self.actions = actions

    def choose_action(self, state: int) -> int:
        """Choose action using epsilon-greedy strategy."""
        if random.random() < self.epsilon:
            return random.randint(0, self.actions - 1)  # Explore
        else:
            return np.argmax(self.q_table[state])  # Exploit

    def learn(self, state: int, action: int, reward: float, next_state: int):
        """Update Q-table based on experience."""
        current_q = self.q_table[state, action]
        max_next_q = np.max(self.q_table[next_state])
        new_q = current_q + self.lr * (reward + self.gamma * max_next_q - current_q)
        self.q_table[state, action] = new_q

    def get_q_table(self):
        """Return current Q-table."""
        return self.q_table

In [ ]:
def simulate_step(state, action):
    """Simulate one step in the grid."""
    row, col = state // 4, state % 4

    if action == 0:    # up
        row = max(0, row - 1)
    elif action == 1:  # right
        col = min(3, col + 1)
    elif action == 2:  # down
        row = min(3, row + 1)
    elif action == 3:  # left
        col = max(0, col - 1)

    next_state = row * 4 + col

    if next_state == 15:   # Goal
        reward = 100
    elif next_state == state:  # Hit wall
        reward = -1
    else:
        reward = -1  # Step penalty

    return next_state, reward

In [ ]:
# Demo: Learning Agent in Simple Grid World
print("\n--- Q-Learning Agent Demo ---")
print("Training agent to navigate a 4x4 grid to reach goal...\n")
agent = QLearningAgent(states=16, actions=4)

episodes = 100
for episode in range(episodes):
    state = 0
    total_reward = 0
    steps = 0

    while state != 15 and steps < 50:
        action = agent.choose_action(state)
        next_state, reward = simulate_step(state, action)
        agent.learn(state, action, reward, next_state)

        state = next_state
        total_reward += reward
        steps += 1

    if (episode + 1) % 20 == 0:
        print(f"Episode {episode + 1}: Total Reward = {total_reward:.1f}, Steps = {steps}")

## Agentic RAG
- Incorporates AI agents into RAG systems
- Multiple specialized agents for different data sources
- Autonomous decision-making about what to retrieve

In [ ]:
class AgenticRAG:
    """
    An agentic RAG system with multiple specialized retrieval agents.
    """
    def __init__(self):
        self.agents = {}

    def register_agent(self, agent_name: str, knowledge_base: List[str]):
        """Register a specialized retrieval agent."""
        self.agents[agent_name] = {
            'knowledge_base': knowledge_base,
            'queries_handled': 0
        }
        print(f"✓ Registered agent: {agent_name}")

    def route_query(self, query: str) -> str:
        """Route query to most appropriate agent."""
        query_lower = query.lower()

        if any(word in query_lower for word in ['cloud', 'scalability', 'deployment']):
            return 'cloud_expert'
        elif any(word in query_lower for word in ['rag', 'retrieval', 'llm']):
            return 'rag_expert'
        elif any(word in query_lower for word in ['agent', 'autonomous', 'learning']):
            return 'agent_expert'
        else:
            return 'cloud_expert'

    def query(self, query: str) -> Dict[str, Any]:
        """Process query through appropriate agent."""
        agent_name = self.route_query(query)

        if agent_name not in self.agents:
            return {'error': 'No suitable agent found'}

        agent = self.agents[agent_name]
        agent['queries_handled'] += 1

        results = self._retrieve_from_agent(query, agent['knowledge_base'])

        return {
            'agent': agent_name,
            'query': query,
            'results': results,
            'confidence': 0.85
        }

    def _retrieve_from_agent(self, query: str, knowledge_base: List[str]) -> List[str]:
        """Retrieve information from agent's knowledge base."""
        query_words = set(query.lower().split())
        scored_docs = []

        for doc in knowledge_base:
            doc_words = set(doc.lower().split())
            intersection = query_words.intersection(doc_words)
            union = query_words.union(doc_words)
            similarity = len(intersection) / len(union) if union else 0
            if similarity > 0:
                scored_docs.append((similarity, doc))

        scored_docs.sort(reverse=True)
        return [doc for _, doc in scored_docs[:2]]

    def get_stats(self):
        """Get statistics about agent usage."""
        stats = {}
        for name, agent in self.agents.items():
            stats[name] = agent['queries_handled']
        return stats

In [ ]:
# Demo: Agentic RAG System
print("\n--- Agentic RAG System Demo ---\n")
agentic_rag = AgenticRAG()

agentic_rag.register_agent('cloud_expert', [
    "Cloud computing provides scalability, resource optimization, and cost efficiency for AI workloads.",
    "Cloud platforms offer serverless execution, global accessibility, and integration with various services.",
    "AI agents in cloud can dynamically allocate resources based on computational demands.",
])
agentic_rag.register_agent('rag_expert', [
    "RAG retrieves facts from external knowledge bases to ground LLM responses in accurate information.",
    "RAG systems combine retrieval mechanisms with generative language models for better answers.",
    "Agentic RAG uses multiple specialized agents to query different types of data sources.",
])
agentic_rag.register_agent('agent_expert', [
    "AI agents are autonomous software entities that perceive, decide, and act to achieve goals.",
    "Learning agents improve through experience using reinforcement learning and adaptation.",
    "Goal-based agents plan ahead and consider future consequences when making decisions.",
])

test_queries = [
    "How does cloud computing support AI agents?",
    "What is RAG and how does it work?",
    "Explain learning agents and their components",
]
for query in test_queries:
    print(f"\n{'='*70}")
    result = agentic_rag.query(query)
    print(f"🤔 Query: {result['query']}")
    print(f"🎯 Routed to: {result['agent']}")
    print(f"📚 Retrieved information:")
    for i, info in enumerate(result['results'], 1):
        print(f"   {i}. {info}")
    print(f"💯 Confidence: {result['confidence']}")

print(f"\n{'='*70}")
print("📊 Agent Usage Statistics:")
stats = agentic_rag.get_stats()
for agent, count in stats.items():
    print(f"   {agent}: {count} queries handled")

In [ ]:
# DiagnosisAgent – Goal-Based + Agentic RAG
# Demonstrates the agent logic from SHARK AgriCloud

from typing import List, Dict, Any

# ── Simulated services (stand-ins for the real SHARK services) ────────────────

class TemperatureService:
    def get_temperature(self, sensor_history: List[Dict]) -> str:
        if not sensor_history:
            return "No sensor data available."
        latest = sensor_history[-1].get("temperature")
        if latest is None:
            return "No temperature reading."
        if latest < 15:
            return f"🔴 {latest}°C – Too cold! Move orchid to a warmer spot."
        elif latest > 30:
            return f"🔴 {latest}°C – Too hot! Provide shade and ventilation."
        else:
            return f"🟢 {latest}°C – Temperature is within optimal range (15–30°C)."

class WateringService:
    def get_recommendation(self, sensor_history: List[Dict]):
        if not sensor_history:
            return "No data available.", "unknown"
        latest = sensor_history[-1].get("humidity", 50)
        if latest < 40:
            return "Soil is dry – water now.", "urgent"
        elif latest < 60:
            return "Soil is slightly dry – water soon.", "watch"
        else:
            return "Soil moisture is fine – no watering needed.", "ok"

class StabilityService:
    def get_stability(self, sensor_history: List[Dict]):
        if not sensor_history:
            return "No data.", None
        in_range = sum(
            1 for r in sensor_history
            if 15 <= r.get("temperature", 0) <= 30
            and 40 <= r.get("humidity", 0) <= 80
        )
        pct = round(in_range / len(sensor_history) * 100)
        return f"{pct}% of readings within normal range.", pct

def run_rag_query(query: str) -> str:
    """Simulated RAG – returns a relevant snippet based on keywords."""
    knowledge_base = [
        "Yellow leaves in orchids are often caused by overwatering or root rot. Ensure proper drainage.",
        "Brown spots on orchid leaves may indicate fungal infection. Remove affected leaves and apply fungicide.",
        "Wilting despite watering suggests root rot – repot in fresh bark medium.",
    ]
    query_words = set(query.lower().split())
    best, best_score = "", 0
    for doc in knowledge_base:
        score = len(query_words & set(doc.lower().split()))
        if score > best_score:
            best, best_score = doc, score
    return best if best_score > 0 else ""


# ── DiagnosisAgent ────────────────────────────────────────────────────────────

class DiagnosisAgent:
    """
    Goal-Based + Agentic RAG agent.
    Receives a natural-language question, routes it to the appropriate tool(s),
    and returns a unified diagnosis report.
    """
    def __init__(self):
        self.temp_service     = TemperatureService()
        self.watering_service = WateringService()
        self.stability_service = StabilityService()

    def run(self, user_input: str, sensor_history: List[Dict]) -> str:
        text    = user_input.lower()
        results = []
        steps   = []

        # Tool 1: Temperature
        temp_keywords = ["temperature", "heat", "cold", "warm", "hot"]
        if any(kw in text for kw in temp_keywords):
            steps.append("🌡️  Tool called: TemperatureService")
            result = self.temp_service.get_temperature(sensor_history)
            results.append(f"**Temperature:** {result}")

        # Tool 2: Watering
        water_keywords = ["water", "watering", "soil", "moisture", "dry", "wet"]
        if any(kw in text for kw in water_keywords):
            steps.append("💧 Tool called: WateringScheduleService")
            rec_text, urgency = self.watering_service.get_recommendation(sensor_history)
            icon = {'urgent': '🔴', 'watch': '🟡', 'ok': '🟢', 'unknown': '⚪'}.get(urgency, '⚪')
            results.append(f"**Watering:** {icon} {rec_text}")

        # Tool 3: RAG – disease/symptom search
        disease_keywords = ["yellow", "brown", "spot", "rot", "disease", "sick", "leaf", "wilt"]
        if any(kw in text for kw in disease_keywords):
            steps.append("🔬 Tool called: RAG Article Search")
            rag_result = run_rag_query(user_input)
            if rag_result:
                results.append(f"**Disease info (from academic articles):**\n{rag_result}")
            else:
                results.append("**Disease info:** No relevant articles found.")

        # Tool 4: Environment stability
        env_keywords = ["environment", "stable", "stability", "conditions", "status", "overall"]
        if any(kw in text for kw in env_keywords):
            steps.append("📊 Tool called: EnvironmentStabilityService")
            stability_text, pct = self.stability_service.get_stability(sensor_history)
            results.append(f"**Environment stability:** {stability_text}")

        # Fallback – no keywords matched → run all tools
        if not results:
            steps.append("⚠️  No specific keywords detected – running all tools")
            results.append(f"**Temperature:** {self.temp_service.get_temperature(sensor_history)}")
            rec_text, urgency = self.watering_service.get_recommendation(sensor_history)
            icon = {'urgent': '🔴', 'watch': '🟡', 'ok': '🟢', 'unknown': '⚪'}.get(urgency, '⚪')
            results.append(f"**Watering:** {icon} {rec_text}")
            stability_text, _ = self.stability_service.get_stability(sensor_history)
            results.append(f"**Environment stability:** {stability_text}")

        print("── Agent reasoning steps ──")
        for s in steps:
            print(" ", s)
        print()
        print("🌸 Agent Diagnosis Report")
        print("─" * 40)
        for r in results:
            print(r)
        return "Done."


In [ ]:
# Demo
sensor_history = [
    {"temperature": 18, "humidity": 35},
    {"temperature": 22, "humidity": 38},
    {"temperature": 25, "humidity": 42},
]

agent = DiagnosisAgent()

print("=== Query 1 ===")
agent.run("The leaves are turning yellow and have brown spots", sensor_history)

print("\n=== Query 2 ===")
agent.run("Should I water the orchid? The soil feels dry.", sensor_history)

print("\n=== Query 3 ===")
agent.run("What is the overall environment status?", sensor_history)
